# dko-3dgs — Reanudación en Colab Pro (IMG_0884, todos los frames)

Retoma la corrida all-frames desde el checkpoint de GLOMAP hecho en local:
**triangulación del modelo COMPLETO (26.557 cámaras)** — que en WSL moría por RAM —
y luego subset, undistorsión y entrenamiento 3DGS.

**Runtime requerido:** `Entorno de ejecución → Cambiar tipo` → **A100 GPU** + **High-RAM**.

**Antes de ejecutar:** sube la carpeta local `~/dko-3dgs/colab_upload/` a tu Drive
como `MyDrive/dko_884/` (debe contener: `IMG_0884.MOV`, `feats-aliked-n16.h5`,
`matches.h5`, `pairs.txt`, `glomap0/`).

Tiempos estimados: copia desde Drive ~10 min · setup ~15 min · frames ~10 min ·
triangulación 1–2 h · entrenamiento ~40 min (A100).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!free -h | head -2
!nproc

## 1. Traer artefactos desde Drive al disco local del runtime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# localizar la carpeta con los artefactos (busca feats-aliked-n16.h5 hasta 2 niveles)
import glob, os
hits = (glob.glob('/content/drive/MyDrive/*/feats-aliked-n16.h5')
        + glob.glob('/content/drive/MyDrive/feats-aliked-n16.h5')
        + glob.glob('/content/drive/MyDrive/*/*/feats-aliked-n16.h5'))
assert hits, 'No encontré feats-aliked-n16.h5 en tu Drive — revisa la subida'
SRC = os.path.dirname(hits[0])
print('carpeta encontrada:', SRC)
!ls -la "$SRC"
# copiar a disco local (los h5 se leen mucho: desde Drive directo sería lentísimo)
!mkdir -p /content/w
!cp -v "$SRC"/feats-aliked-n16.h5 "$SRC"/matches.h5 "$SRC"/pairs.txt /content/w/
!cp -rv "$SRC"/glomap0 /content/w/
!cp -v "$SRC"/IMG_0884.MOV /content/w/

## 2. Instalar hloc + pycolmap + gaussian-splatting

In [ ]:
!git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization /content/hloc_repo
!pip install -q -e /content/hloc_repo
!pip install -q pycolmap==4.1.1 || pip install -q pycolmap
import pycolmap
print('pycolmap', pycolmap.__version__)

In [ ]:
!git clone --quiet --recursive https://github.com/graphdeco-inria/gaussian-splatting /content/gaussian-splatting
!pip install -q plyfile \
    /content/gaussian-splatting/submodules/diff-gaussian-rasterization \
    /content/gaussian-splatting/submodules/simple-knn \
    /content/gaussian-splatting/submodules/fused-ssim

## 3. Extraer los frames del video (los nombres deben calzar: 00001.jpg...)

In [ ]:
!mkdir -p /content/frames
!ffmpeg -hide_banner -loglevel error -i /content/w/IMG_0884.MOV -qscale:v 2 /content/frames/%05d.jpg
from pathlib import Path
print(len(list(Path('/content/frames').glob('*.jpg'))), 'frames')

## 4. Triangulación del modelo COMPLETO (26.557 cámaras)

El paso que en WSL (30 GB) moría por OOM. Con High-RAM debería caber —
vigila el medidor de RAM del runtime durante esta celda.

In [ ]:
import pycolmap
from pathlib import Path
from hloc import triangulation

W = Path('/content/w')
images = Path('/content/frames')
glomap_model = W / 'glomap0'
sfm_tri = Path('/content/tri')

# CHECKPOINT: si ya hay una triangulación guardada en Drive, reutilizarla
ck = Path(SRC) / 'ckpt_tri'
if (ck / 'points3D.bin').exists():
    print('[ckpt] triangulación encontrada en Drive — copiando en vez de recalcular')
    !mkdir -p "$sfm_tri"
    !cp "$ck"/*.bin /content/tri/
    rec = pycolmap.Reconstruction(str(sfm_tri))
else:
    best = pycolmap.Reconstruction(str(glomap_model))
    print(f'modelo GLOMAP: {best.num_reg_images()} registradas, {best.num_points3D()} puntos')

    reg_names = {best.image(i).name for i in best.reg_image_ids()}
    pairs_tri = W / 'pairs_tri.txt'
    kept = [l for l in (W / 'pairs.txt').read_text().splitlines()
            if (p := l.split()) and len(p) == 2 and p[0] in reg_names and p[1] in reg_names]
    pairs_tri.write_text('\n'.join(kept) + '\n')
    print(f'{len(kept)} pares para triangular')

    rec = triangulation.main(
        sfm_tri, glomap_model, images, pairs_tri,
        W / 'feats-aliked-n16.h5', W / 'matches.h5',
    )
    # respaldar de inmediato a Drive (checkpoint de 1-2h de trabajo)
    !mkdir -p "$ck"
    !cp /content/tri/*.bin "$ck"/
    print('checkpoint de triangulación respaldado en Drive')

print(f'triangulado: {rec.num_points3D()} puntos, '
      f'err {rec.compute_mean_reprojection_error():.3f}px')

## 5. Subset de entrenamiento + undistort

Con A100 (40 GB VRAM) + High-RAM entrenamos con **4000 cámaras** a `-r 4`.
La nube inicial usa TODOS los puntos del modelo completo triangulado.

In [ ]:
import shutil

MAX_TRAIN = 4000

reg = sorted(rec.reg_image_ids())
step = max(1, len(reg) // MAX_TRAIN)
keep = set(reg[::step])
for iid in reg:
    if iid not in keep:
        rec.deregister_frame(rec.image(iid).frame_id)
sfm_train = Path('/content/sfm_train')
sfm_train.mkdir(exist_ok=True)
rec.write(str(sfm_train))
print(f'set de entrenamiento: {len(keep)} cámaras (1 de cada {step})')

data = Path('/content/data')
pycolmap.undistort_images(output_path=str(data), input_path=str(sfm_train),
                          image_path=str(images), output_type='COLMAP')
sparse0 = data / 'sparse' / '0'
sparse0.mkdir(parents=True, exist_ok=True)
for f in (data / 'sparse').iterdir():
    if f.is_file():
        shutil.move(str(f), sparse0 / f.name)
# nube completa del modelo triangulado (3DGS solo lee xyz/rgb)
shutil.copy(Path('/content/tri') / 'points3D.bin', sparse0 / 'points3D.bin')
print(sorted(p.name for p in sparse0.iterdir()))

## 6. Entrenamiento 3DGS

In [ ]:
%cd /content/gaussian-splatting
!python train.py -s /content/data -m /content/output/dko3d_884all \
    -r 4 --data_device cpu \
    --iterations 30000 --save_iterations 30000 --test_iterations -1 \
    --checkpoint_iterations 10000 20000

## 7. Guardar el resultado en Drive

In [ ]:
import os
DEST = os.path.join(SRC, 'resultado')
!mkdir -p "$DEST"
!cp -r /content/output/dko3d_884all/point_cloud "$DEST"/
!cp /content/output/dko3d_884all/cameras.json /content/output/dko3d_884all/cfg_args "$DEST"/ 2>/dev/null
# checkpoints de entrenamiento por si quieres continuar/reentrenar después
!cp /content/output/dko3d_884all/chkpnt*.pth "$DEST"/ 2>/dev/null
print('Modelo guardado en', DEST + '/point_cloud/iteration_30000/point_cloud.ply')